<a href="https://colab.research.google.com/github/MARVINTIEGO/AAI-2026/blob/main/Exercise1_PromptChaining.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [21]:
import re, json

def customer_support_bot(prompt: str) -> str:
    p = prompt.lower()

    # STEP 2 FIRST (so it doesn't get overridden by Step 1 detection)
    if "step 2" in p:

        # Weak prompt behavior
        if ("only" not in p) or ("max 3" not in p):
            return (
                "Can you provide more details so I can help?\n"
                "- What exactly is wrong with the laptop?\n"
                "- When did you buy it?\n"
                "- What is the email on the account?\n"
                "- Do you want a refund or exchange?"
            )

        # Strong prompt behavior
        try:
            data = json.loads(re.search(r"\{.*\}", prompt, re.DOTALL).group(0))
            items = data.get("missing_info", [])[:3]
        except Exception:
            items = ["order number", "purchase date", "account email"]

        bullets = "\n".join([f"- Could you share your {it}?" for it in items])
        return (
            "I’m sorry that happened — I’ll help you get this resolved.\n"
            f"{bullets}\n"
            "Once I have that, I can guide you through the quickest next steps."
        )

    # STEP 1
    if "step 1" in p and ("classify" in p or "triage" in p):
        order_related = "order" in p or "arrived" in p or "damaged" in p
        account_related = "log in" in p or "login" in p or "password" in p

        missing = []
        if order_related: missing.append("order number")
        missing.append("purchase date")
        if account_related: missing.append("account email")

        out = {
            "primary_issue": "damaged_item" if order_related else "unknown",
            "secondary_issues": ["account_access"] if account_related else [],
            "sentiment": "negative",
            "entities": {"order_related": order_related, "account_related": account_related},
            "missing_info": missing
        }
        return json.dumps(out, indent=2)

    # STEP 3
    if "step 3" in p:
        return (
            "1) Confirm your order number and purchase date.\n"
            "2) Take photos of the damage.\n"
            "3) Start a return under 'Damaged on Arrival'.\n"
            "4) Use 'Forgot Password' if login fails.\n"
            "5) Check spam for reset email.\n"
            "6) Contact support if reset fails.\n"
            "If this doesn’t work… we will escalate to an agent."
        )

    # STEP 4
    if "step 4" in p:
        return json.dumps(
            {"escalate": True, "reason": "Account access blocks return process.", "priority": "medium"},
            indent=2
        )

    return "Simulated response."

In [23]:
customer_message = "My order arrived damaged and I can't log in to start a return."

# Step 1: Classify
step1_prompt = f"""
STEP 1 — TRIAGE / CLASSIFY
You are a triage agent for an e-commerce electronics store.
Classify the issue(s) and extract missing information.
Output valid JSON only.

Customer message:
{customer_message}
"""
step1_output = customer_support_bot(step1_prompt)
print("STEP 1 OUTPUT:\n", step1_output)

# Step 2: Ask for missing info (uses Step 1 output)
step2_prompt = f"""
STEP 2 — GATHER MISSING INFO
You are a polite customer support agent.
Ask ONLY for the missing information listed in the Step 1 JSON.
Ask max 3 questions as bullet points and end with reassurance.

Step 1 JSON:
{step1_output}

Original customer message:
{customer_message}
"""
step2_output = customer_support_bot(step2_prompt)
print("\nSTEP 2 OUTPUT:\n", step2_output)

# Simulated customer answers (you can edit)
customer_answers = """
- Order number: 123-4567890
- Purchase date: Feb 10, 2026
- Account email: student@example.com
""".strip()

# Step 3: Solution (uses Step 1 + answers)
step3_prompt = f"""
STEP 3 — PROPOSE SOLUTION
You are a senior support agent.
Provide a numbered step-by-step solution (5–8 steps).
Avoid blaming language.
Include one 'If this doesn't work...' fallback.

Step 1 JSON:
{step1_output}

Customer answers:
{customer_answers}

Original message:
{customer_message}
"""
step3_output = customer_support_bot(step3_prompt)
print("\nSTEP 3 OUTPUT:\n", step3_output)

# Step 4: Escalation decision
step4_prompt = f"""
STEP 4 — ESCALATION DECISION
You are an escalation bot.
Output JSON only:
{{"escalate": true/false, "reason": "string", "priority": "low|medium|high"}}

Escalate if unresolved or customer cannot proceed due to access issues.

Step 1 JSON:
{step1_output}

Solution:
{step3_output}
"""
step4_output = customer_support_bot(step4_prompt)
print("\nSTEP 4 OUTPUT:\n", step4_output)

STEP 1 OUTPUT:
 {
  "primary_issue": "damaged_item",
  "secondary_issues": [
    "account_access"
  ],
  "sentiment": "negative",
  "entities": {
    "order_related": true,
    "account_related": true
  },
  "missing_info": [
    "order number",
    "purchase date",
    "account email"
  ]
}

STEP 2 OUTPUT:
 I’m sorry that happened — I’ll help you get this resolved.
- Could you share your order number?
- Could you share your purchase date?
- Could you share your account email?
Once I have that, I can guide you through the quickest next steps.

STEP 3 OUTPUT:
 1) Confirm your order number and purchase date.
2) Take photos of the damage.
3) Start a return under 'Damaged on Arrival'.
4) Use 'Forgot Password' if login fails.
5) Check spam for reset email.
6) Contact support if reset fails.
If this doesn’t work… we will escalate to an agent.

STEP 4 OUTPUT:
 {
  "escalate": true,
  "reason": "Account access blocks return process.",
  "priority": "medium"
}


In [25]:
# --- Iteration Evidence (Before vs After) ---

# BEFORE: vague prompt (no constraints)
step2_prompt_before = f"""
STEP 2
Ask the customer what you need.

Step 1 JSON:
{step1_output}
"""
before_out = customer_support_bot(step2_prompt_before)

# AFTER: improved prompt (constraints)
step2_prompt_after = f"""
STEP 2 — GATHER MISSING INFO
Ask ONLY for the missing information listed in the Step 1 JSON.
Ask max 3 questions as bullet points and end with reassurance.

Step 1 JSON:
{step1_output}

Original customer message:
{customer_message}
"""
after_out = customer_support_bot(step2_prompt_after)

print("STEP 2 BEFORE (weaker prompt):\n", before_out)
print("\nSTEP 2 AFTER (improved prompt):\n", after_out)

STEP 2 BEFORE (weaker prompt):
 Can you provide more details so I can help?
- What exactly is wrong with the laptop?
- When did you buy it?
- What is the email on the account?
- Do you want a refund or exchange?

STEP 2 AFTER (improved prompt):
 I’m sorry that happened — I’ll help you get this resolved.
- Could you share your order number?
- Could you share your purchase date?
- Could you share your account email?
Once I have that, I can guide you through the quickest next steps.
